In [ ]:
# Import necessary libraries
import os
import shutil
import cv2
import numpy as np
import yaml
from sklearn.model_selection import StratifiedKFold
from ultralytics import YOLO
from albumentations import Compose, RandomBrightnessContrast, HueSaturationValue, CLAHE, Rotate
from tqdm import tqdm

# Define paths and parameters
image_dir = 'dataset/20x_Unstained'
annotation_dir = 'dataset/yoloAnnotations'
output_dir = 'detectTest/mediumV25'
k = 3   # Reduce the number of folds
imgsz = 1312  # Image size
epochs = 100  # Increase epochs
batch_size = 4  # Batch size
conf_threshold = 0.5  # Confidence threshold for validation

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Function to compute brightness of an image
def compute_brightness(image_path):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    return np.mean(image)

# Data augmentation pipeline
def augment_image(image_path, output_path):
    image = cv2.imread(image_path)
    if image is None:
        print(f"Warning: Could not read image {image_path}")
        return False

    augmentations = Compose([
        RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.7),
        HueSaturationValue(hue_shift_limit=15, sat_shift_limit=20, val_shift_limit=15, p=0.7),
        CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=0.7),
        Rotate(limit=15, p=0.5)
    ])

    try:
        augmented = augmentations(image=image)
        augmented_image = augmented['image']
        cv2.imwrite(output_path, augmented_image)
        return True
    except Exception as e:
        print(f"Error augmenting image {image_path}: {e}")
        return False

# Function to create YAML file for YOLO training
def create_yaml_file(fold_index, train_files, val_files):
    fold_dir = os.path.join(output_dir, f'fold_{fold_index}')
    os.makedirs(os.path.join(fold_dir, 'images', 'train'), exist_ok=True)
    os.makedirs(os.path.join(fold_dir, 'images', 'val'), exist_ok=True)
    os.makedirs(os.path.join(fold_dir, 'labels', 'train'), exist_ok=True)
    os.makedirs(os.path.join(fold_dir, 'labels', 'val'), exist_ok=True)

    # Copy and preprocess files
    for file in tqdm(train_files, desc="Processing training files"):
        base_filename = os.path.splitext(file)[0]
        label_file = f"{base_filename}.txt"

        # Original image - copy to training folder
        original_img_path = os.path.join(fold_dir, 'images', 'train', file)
        shutil.copy2(os.path.join(image_dir, file), original_img_path)

        # Copy corresponding label file
        if os.path.exists(os.path.join(annotation_dir, label_file)):
            shutil.copy2(
                os.path.join(annotation_dir, label_file),
                os.path.join(fold_dir, 'labels', 'train', label_file)
            )

        # Create augmented version
        aug_img_name = f"{base_filename}_aug.png"
        aug_img_path = os.path.join(fold_dir, 'images', 'train', aug_img_name)

        if augment_image(os.path.join(image_dir, file), aug_img_path):
            # Copy the label file for the augmented image
            if os.path.exists(os.path.join(annotation_dir, label_file)):
                shutil.copy2(
                    os.path.join(annotation_dir, label_file),
                    os.path.join(fold_dir, 'labels', 'train', f"{base_filename}_aug.txt")
                )

    for file in tqdm(val_files, desc="Processing validation files"):
        # Copy validation images without augmentation
        shutil.copy2(
            os.path.join(image_dir, file),
            os.path.join(fold_dir, 'images', 'val', file)
        )

        # Copy corresponding label file
        label_file = file.replace('.png', '.txt')
        if os.path.exists(os.path.join(annotation_dir, label_file)):
            shutil.copy2(
                os.path.join(annotation_dir, label_file),
                os.path.join(fold_dir, 'labels', 'val', label_file)
            )

    # Create YAML file
    yaml_content = {
        'path': os.path.abspath(fold_dir),  # Dataset root dir
        'train': os.path.join('images', 'train'),  # Train images relative to 'path'
        'val': os.path.join('images', 'val'),  # Val images relative to 'path'
        'nc': 1,  # Number of classes
        'names': ['wbc']  # Class names
    }

    yaml_path = os.path.join(fold_dir, 'data.yaml')
    with open(yaml_path, 'w') as f:
        yaml.dump(yaml_content, f, sort_keys=False)

    return yaml_path

# Main K-Fold Cross-Validation and Training Loop
if __name__ == "__main__":
    # Get list of image files and compute brightness
    image_files = sorted([f for f in os.listdir(image_dir) if f.endswith('.png')])
    brightness_values = [compute_brightness(os.path.join(image_dir, f)) for f in image_files]

    # Create bins for stratification based on brightness
    brightness_bins = np.digitize(brightness_values, bins=np.linspace(0, 255, 10))

    # Stratified K-Fold splitting based on brightness
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=55)

    for fold_index, (train_index, val_index) in enumerate(skf.split(image_files, brightness_bins)):
        print(f"Processing fold {fold_index + 1}/{k}")

        # Get train and validation files
        train_files = [image_files[i] for i in train_index]
        val_files = [image_files[i] for i in val_index]

        # Create YAML file and get its path
        yaml_path = create_yaml_file(fold_index, train_files, val_files)

        # Train YOLO model
        model = YOLO('yolov8m.pt')  # Use YOLO medium model
        model.train(
            data=yaml_path,
            epochs=epochs,
            imgsz=imgsz,
            batch=batch_size,
            amp=True,
            patience=20,
            mosaic=1.0,
            mixup=0.5,
            copy_paste=0.5,
            optimizer='AdamW',
            momentum=0.937,
            weight_decay=0.0005,
            dropout=0.1,
            lr0=0.0005,
            lrf=0.01,
            warmup_epochs=10,
            cos_lr=True,
            project=output_dir,
            name=f'fold_{fold_index}_results',
            verbose=True,
            box=7.5,
            cls=0.5,
            dfl=1.5,
            label_smoothing=0.1,
            conf=conf_threshold,
            single_cls=True
        )

    print("K-fold cross-validation training complete.")

Processing fold 1/3


Processing validation files: 100%|██████████████| 16/16 [00:01<00:00,  9.06it/s]


New https://pypi.org/project/ultralytics/8.3.109 available 😃 Update with 'pip install -U ultralytics'
WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in in the future.
Ultralytics 8.3.85 🚀 Python-3.10.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce GTX 1080, 8111MiB)
engine/trainer: task=detect, mode=train, model=yolov8m.pt, data=detectTest/mediumV25/fold_0/data.yaml, epochs=100, time=None, patience=20, batch=4, imgsz=1300, save=True, save_period=-1, cache=False, device=None, workers=8, project=detectTest/mediumV25, name=fold_0_results2, exist_ok=False, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=True, rect=False, cos_lr=True, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.1, val=True, split=val, save_json=False, save_hybrid=False, conf=0.5, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer

train: Scanning /home/dbh3/Documents/FYP/detectTest/mediumV25/fold_0/labels/trai

train: New cache created: /home/dbh3/Documents/FYP/detectTest/mediumV25/fold_0/labels/train.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


...

     85/100      6.55G      1.218      1.019     0.9501         29       1312: 1
                 Class     Images  Instances      Box(P          R      mAP50  m


                   all         16        235      0.982      0.949       0.97      0.679

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     86/100      6.62G      1.148     0.8747      0.933         41       1312: 1
                 Class     Images  Instances      Box(P          R      mAP50  m


                   all         16        235       0.99      0.953      0.974      0.681

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     87/100      6.53G       1.18     0.9326     0.9417        125       1312:  